# HR Data Collection, Cleaning and Validation

**Project type:** Simulated HR employee dataset  
**Purpose:** Demonstrate an auditable, repeatable data preparation workflow.  

The raw file intentionally includes missing values, inconsistent category casing, invalid values, malformed dates and exact duplicate rows. All data is synthetic and must not be treated as real employee information.

## 1. Import libraries and inspect the raw data
Load the CSV, check dimensions, preview records, inspect data types and quantify missingness.

In [ ]:
import pandas as pd
import numpy as np

raw = pd.read_csv('HR_Raw_Dataset.csv')
print('Shape:', raw.shape)
display(raw.head())
print('\nData types:\n', raw.dtypes)
print('\nMissing values:\n', raw.isna().sum())

## 2. Assess data quality
Check exact duplicates and inspect category values. Invalid numeric ranges and malformed dates are treated as data quality issues.

In [ ]:
print('Exact duplicate rows:', raw.duplicated().sum())
for col in ['Department', 'Employment_Status', 'Gender']:
    print(f'\n{col}:')
    print(raw[col].value_counts(dropna=False))

## 3. Clean the data
Remove exact duplicates; standardize text; coerce numeric/date columns; convert out-of-range values to missing; impute numeric values using medians and categorical values using `Unknown`. Missing joining dates are filled with the median valid date. These are explicit project assumptions, not universal HR policies.

In [ ]:
# HR Data Collection, Cleaning and Validation
# Reproducible workflow: run from the folder containing HR_Raw_Dataset.csv.

import pandas as pd
import numpy as np

# 1. Import the raw employee data.
df = pd.read_csv("HR_Raw_Dataset.csv")
print("Raw shape:", df.shape)
print("Missing values before cleaning:\n", df.isna().sum())

# 2. Remove exact duplicate rows.
df = df.drop_duplicates().copy()

# 3. Standardize text fields (trim spaces and normalize casing).
text_cols = ["Employee_Name", "Gender", "Department", "Job_Role",
             "Location", "Employment_Status"]
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

# Canonical category labels.
df["Department"] = df["Department"].str.title()
df["Employment_Status"] = df["Employment_Status"].str.title()
df["Gender"] = df["Gender"].replace({
    "Non-Binary": "Non-binary",
    "Prefer Not To Say": "Prefer not to say"
})

# 4. Convert columns to appropriate numeric/date types.
numeric_cols = ["Age", "Annual_Salary_INR", "Years_of_Experience",
                "Performance_Rating", "Attendance_Percentage", "Leave_Days"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["Date_of_Joining"] = pd.to_datetime(df["Date_of_Joining"], errors="coerce")

# 5. Identify invalid values and convert them to missing.
# These bounds are project assumptions for this simulated dataset.
df.loc[~df["Age"].between(18, 65), "Age"] = np.nan
df.loc[df["Annual_Salary_INR"] <= 0, "Annual_Salary_INR"] = np.nan
df.loc[~df["Years_of_Experience"].between(0, 45), "Years_of_Experience"] = np.nan
df.loc[~df["Performance_Rating"].between(1, 5), "Performance_Rating"] = np.nan
df.loc[~df["Attendance_Percentage"].between(0, 100), "Attendance_Percentage"] = np.nan
df.loc[~df["Leave_Days"].between(0, 365), "Leave_Days"] = np.nan
df.loc[df["Date_of_Joining"].isna(), "Date_of_Joining"] = pd.NaT

# 6. Handle missing values.
# Median is robust to skew/outliers for continuous numeric fields.
for col in ["Age", "Annual_Salary_INR", "Years_of_Experience",
            "Attendance_Percentage", "Leave_Days"]:
    df[col] = df[col].fillna(df[col].median())

# Rating is ordinal: use median, then round to an integer.
df["Performance_Rating"] = df["Performance_Rating"].fillna(
    df["Performance_Rating"].median()
).round().astype("int64")

# Categorical fields: explicit "Unknown" avoids inventing a category.
for col in ["Gender", "Department", "Job_Role", "Location", "Employment_Status"]:
    df[col] = df[col].fillna("Unknown")

# Invalid/missing join dates are imputed with the median valid date.
median_join_date = df["Date_of_Joining"].dropna().sort_values().iloc[len(df["Date_of_Joining"].dropna()) // 2]
df["Date_of_Joining"] = df["Date_of_Joining"].fillna(median_join_date)

# 7. Enforce data types and consistent formats.
df["Age"] = df["Age"].round().astype("int64")
df["Annual_Salary_INR"] = df["Annual_Salary_INR"].round().astype("int64")
df["Years_of_Experience"] = df["Years_of_Experience"].round().astype("int64")
df["Leave_Days"] = df["Leave_Days"].round().astype("int64")
df["Attendance_Percentage"] = df["Attendance_Percentage"].round(1)
df["Date_of_Joining"] = df["Date_of_Joining"].dt.strftime("%Y-%m-%d")

# 8. Validate key rules.
assert df["Employee_ID"].notna().all(), "Employee_ID contains missing values"
assert df["Employee_ID"].is_unique, "Employee_ID should be unique"
assert df["Age"].between(18, 65).all()
assert df["Annual_Salary_INR"].gt(0).all()
assert df["Performance_Rating"].between(1, 5).all()
assert df["Attendance_Percentage"].between(0, 100).all()
assert df["Leave_Days"].ge(0).all()
assert df.isna().sum().sum() == 0, "Unexpected missing values remain"

# 9. Export cleaned dataset and validation summary.
df.to_csv("HR_Cleaned_Dataset.csv", index=False)
summary = df.describe(include="all").transpose()
summary.to_csv("HR_Validation_Summary.csv")
print("Cleaned shape:", df.shape)
print("Missing values after cleaning:\n", df.isna().sum())
print("Department counts:\n", df["Department"].value_counts())
print("Validation passed. Cleaned file exported.")


## 4. Validate and summarize
The assertions verify uniqueness, non-null IDs, allowed ranges and zero remaining missing values. The final summary supports a high-level review.

In [ ]:
print('Cleaned shape:', df.shape)
print('Missing values after cleaning:\n', df.isna().sum())
print('\nNumeric summary:')
display(df.describe())
print('\nDepartment distribution:')
display(df['Department'].value_counts().rename_axis('Department').to_frame('Employees'))
print('\nEmployment status distribution:')
display(df['Employment_Status'].value_counts().rename_axis('Status').to_frame('Employees'))

## 5. Export outputs
The cleaned CSV and validation summary are saved in the current working directory. Keep the raw file unchanged for auditability.

In [ ]:
df.to_csv('HR_Cleaned_Dataset.csv', index=False)
df.describe(include='all').transpose().to_csv('HR_Validation_Summary.csv')
print('Exports completed.')

## 6. Limitations and responsible use
This is synthetic demonstration data. Median imputation can reduce variation and may not be appropriate for production HR decisions. In a real organization, confirm definitions and permissible ranges with HR/data owners, retain missingness flags, document approvals, protect personal data, and avoid using imputed values for consequential employment decisions.